#02_silver_app_logs.sql

Camada Silver - fact_app_logs

Este notebook prepara logs de eventos do app bronze.app_event_log para consumo analítico e monitoramento (experiência digital, estabilidade, erros, performance), entrega uma tabela silver tipada, deduplicada e rastreável: silver.fact_app_logs.

##Objetivos
- tipa timestamp_evento (TIMESTAMP) e métricas (http_status / latencia_ms)
- padroniza texto (canal / evento)
- deduplica por event_id mantendo o registro mais recente por ingestion_ts
- rejeita apenas erros estruturais do evento (event_id, timestamp_evento, latencia_ms)
- persiste idempotente via MERGE usando row_hash
- roda incremental via checkpoint (silver_ops.pipeline_checkpoint)

##Estratégia:
- SQL-first com TEMP VIEWs
- TRY_CAST / TRY_TO_TIMESTAMP para conversões resilientes
- row_number() para dedupe determinístico
- http_status inválido não é reject, vira NULL
- MERGE com row_hash

##Chave natural 

event_id

##Contexto

In [0]:
USE CATALOG healthcare_dev;

##Cria schemas

In [0]:
CREATE SCHEMA IF NOT EXISTS healthcare_dev.silver;
CREATE SCHEMA IF NOT EXISTS healthcare_dev.silver_rejects;
CREATE SCHEMA IF NOT EXISTS healthcare_dev.silver_ops;

##Pipeline checkpoint

In [0]:
CREATE TABLE IF NOT EXISTS healthcare_dev.silver_ops.pipeline_checkpoint (
  table_name STRING,
  last_ingestion_ts TIMESTAMP,
  last_run_ts TIMESTAMP,
  status STRING
)
USING DELTA;

##Tabelas de destino (silver + rejects)

In [0]:
CREATE TABLE IF NOT EXISTS healthcare_dev.silver.fact_app_logs (
  event_id STRING,
  beneficiario_id STRING,
  canal STRING,
  evento STRING,
  http_status INT,
  latencia_ms INT,
  sessao_id STRING,
  timestamp_evento TIMESTAMP,
  ingestion_ts TIMESTAMP,
  load_id STRING,
  source_file STRING,
  row_hash STRING
)
USING DELTA;

In [0]:
CREATE TABLE IF NOT EXISTS healthcare_dev.silver_rejects.fact_app_logs (
  event_id STRING,
  beneficiario_id STRING,
  canal STRING,
  evento STRING,
  http_status STRING,
  latencia_ms STRING,
  sessao_id STRING,
  timestamp_evento STRING,
  ingestion_ts TIMESTAMP,
  load_id STRING,
  source_file STRING,
  reject_reason STRING,
  reject_ts TIMESTAMP,
  http_status_raw STRING,
  flag_invalid_http_status INT
)
USING DELTA;

##Leitura incremental do bronze (via checkpoint)

In [0]:
CREATE OR REPLACE TEMP VIEW stg_app_logs_raw AS
WITH last_cp AS (
  SELECT coalesce(max(last_ingestion_ts), timestamp('1900-01-01')) AS last_ingestion_ts
  FROM healthcare_dev.silver_ops.pipeline_checkpoint
  WHERE table_name = 'fact_app_logs'
)
SELECT *
FROM healthcare_dev.bronze.app_event_log
WHERE ingestion_ts >= (SELECT last_ingestion_ts FROM last_cp) - INTERVAL 3 DAYS;

##Tipagem e padronização

In [0]:
CREATE OR REPLACE TEMP VIEW stg_app_logs_typed AS
SELECT
  cast(event_id as string) AS event_id,
  cast(beneficiario_id as string) AS beneficiario_id,
  upper(trim(cast(canal as string))) AS canal,
  upper(trim(cast(evento as string))) AS evento,

  -- mantém o raw para auditoria
  cast(http_status as string) AS http_status_raw,

  -- tentativa numérica
  try_cast(http_status as int) AS http_status_num,

  try_cast(latencia_ms as int) AS latencia_ms,
  cast(sessao_id as string) AS sessao_id,
  try_to_timestamp(timestamp_evento) AS timestamp_evento,
  ingestion_ts,
  load_id,
  source_file
FROM stg_app_logs_raw;

##Deduplicação determinística (event_id)

In [0]:
CREATE OR REPLACE TEMP VIEW stg_app_logs_dedup AS
SELECT *
FROM (
  SELECT
    *,
    row_number() OVER (
      PARTITION BY event_id
      ORDER BY ingestion_ts DESC
    ) AS rn
  FROM stg_app_logs_typed
) x
WHERE rn = 1;

##Regras de qualidade (classificação + motivo)

In [0]:
CREATE OR REPLACE TEMP VIEW stg_app_logs_classified AS
SELECT
  *,
  CASE
    WHEN http_status_num IS NOT NULL AND (http_status_num < 100 OR http_status_num > 599) THEN 1
    WHEN http_status_num IS NULL AND http_status_raw IS NOT NULL AND trim(http_status_raw) <> '' THEN 1
    ELSE 0
  END AS flag_invalid_http_status,

  CASE
    WHEN event_id IS NULL OR event_id = '' THEN 'MISSING_EVENT_ID'
    WHEN timestamp_evento IS NULL THEN 'INVALID_TIMESTAMP_EVENTO'
    WHEN latencia_ms IS NOT NULL AND latencia_ms < 0 THEN 'NEGATIVE_LATENCIA_MS'
    ELSE NULL
  END AS reject_reason,

  -- http_status_clean: só fica se estiver na faixa, senão é NULL
  CASE
    WHEN http_status_num BETWEEN 100 AND 599 THEN http_status_num
    ELSE NULL
  END AS http_status_clean
FROM stg_app_logs_dedup;

##Persiste rejects

In [0]:
INSERT INTO healthcare_dev.silver_rejects.fact_app_logs
SELECT
  event_id,
  beneficiario_id,
  canal,
  evento,
  cast(http_status as string) AS http_status,
  cast(latencia_ms as string) AS latencia_ms,
  sessao_id,
  cast(timestamp_evento as string) AS timestamp_evento,
  ingestion_ts,
  load_id,
  source_file,
  reject_reason,
  current_timestamp() AS reject_ts
FROM stg_app_logs_classified
WHERE reject_reason IS NOT NULL;

##Valid + row_hash (para MERGE idempotente) http_status_clean + flag

In [0]:
CREATE OR REPLACE TEMP VIEW stg_app_logs_valid AS
SELECT
  event_id,
  beneficiario_id,
  canal,
  evento,
  http_status_clean AS http_status,
  latencia_ms,
  sessao_id,
  timestamp_evento,
  ingestion_ts,
  load_id,
  source_file,
  http_status_raw,
  flag_invalid_http_status,
  sha2(concat_ws('||',
    event_id,
    coalesce(beneficiario_id,''),
    coalesce(canal,''),
    coalesce(evento,''),
    coalesce(cast(http_status_clean as string),''),
    coalesce(cast(latencia_ms as string),''),
    coalesce(sessao_id,''),
    coalesce(cast(timestamp_evento as string),''),
    coalesce(http_status_raw,''),
    coalesce(cast(flag_invalid_http_status as string),'')
  ), 256) AS row_hash
FROM stg_app_logs_classified
WHERE reject_reason IS NULL;

##MERGE na silver

In [0]:
MERGE INTO healthcare_dev.silver.fact_app_logs AS tgt
USING stg_app_logs_valid AS src
ON tgt.event_id = src.event_id
WHEN MATCHED AND tgt.row_hash <> src.row_hash THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

##Atualiza checkpoint

In [0]:
INSERT INTO healthcare_dev.silver_ops.pipeline_checkpoint
SELECT
  'fact_app_logs' AS table_name,
  max(ingestion_ts) AS last_ingestion_ts,
  current_timestamp() AS last_run_ts,
  'SUCCESS' AS status
FROM stg_app_logs_valid;

##Checa sanidade

In [0]:
SELECT COUNT(*) AS n_silver_app_logs
FROM healthcare_dev.silver.fact_app_logs;

In [0]:
SELECT COUNT(*) AS n_rejects_app_logs
FROM healthcare_dev.silver_rejects.fact_app_logs;

In [0]:
-- Distribuição de motivos
SELECT reject_reason, COUNT(*) AS qtd
FROM healthcare_dev.silver_rejects.fact_app_logs
GROUP BY reject_reason
ORDER BY qtd DESC;